# 🧹 Lesson 04: Cleaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/04_cleaning.ipynb)

> **Garbage in, garbage out. If the text is broken, the search and the answers will be broken too.**

---

## 🎯 Goal
By the end of this lesson you can find defects in extracted text and fix them with small, explicit, testable rules.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~20 minutes |
| 📚 **Prerequisites** | [Lesson 03](03_parsing.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `pdfplumber`, `langchain-core` |

In [ ]:
!pip install -q -U pdfplumber langchain-core

In [1]:
from pathlib import Path
import urllib.request
from collections import Counter
import pdfplumber
from langchain_core.documents import Document

PDF_URL = "https://raw.githubusercontent.com/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/main/data/pdf1.pdf"

pdf_path = next((p for p in [Path("data/pdf1.pdf"), Path("../data/pdf1.pdf")] if p.exists()), None)
if pdf_path is None:
    pdf_path = Path("data/pdf1.pdf")
    pdf_path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PDF_URL, pdf_path)

lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        for line in page.extract_text_lines():
            size = round(max(char["size"] for char in line["chars"]), 1)
            lines.append({"text": line["text"], "size": size, "page": page_number})

body_size = Counter(line["size"] for line in lines).most_common(1)[0][0]

sections = []
current = None
for line in lines:
    if line["size"] > body_size:
        current = {"title": line["text"], "page": line["page"], "text": []}
        sections.append(current)
    elif current is not None:
        current["text"].append(line["text"])
sections = [section for section in sections if section["text"]]

parsed_documents = [
    Document(
        page_content=" ".join(section["text"]),
        metadata={"source": pdf_path.name, "page": section["page"], "section": section["title"]},
    )
    for section in sections
]

print("documents:", len(parsed_documents))

documents: 4


## 🔍 Step 1: Find the defects

We never clean blindly. First we **look**, and write down what is wrong. Otherwise we risk "fixing" things that were fine.

Let's search our documents for the defects we noticed in Lessons 02 and 03.

In [2]:
import re

for doc in parsed_documents:
    text = doc.page_content
    cid_codes = re.findall(r"\(cid:\d+\)", text)
    full_time = re.findall(r"Full\Wtime|Fulln?time", text)
    odd_chars = sorted({char for char in text if ord(char) > 127})
    if cid_codes or full_time or odd_chars:
        print(doc.metadata["section"])
        print("   cid codes:", cid_codes)
        print("   Full-time variants:", full_time)
        print("   non-ASCII:", odd_chars)

Paid Time Off (PTO) Policy
   cid codes: ['(cid:127)', '(cid:127)', '(cid:127)']
   Full-time variants: ['Fullntime']
   non-ASCII: ['–']


## 🛠️ Step 2: Write the cleaning rules

Each defect gets **one small rule**. Small rules are easy to read, test, and remove if they turn out to be wrong.

| Defect | Rule |
|---|---|
| Different ways to write the same character | Unicode **normalization** (`NFKC`) |
| `(cid:127)` where a bullet should be | replace with `•` on a new line |
| `Fullntime` / `Full■time` | replace with `Full-time` |
| Extra spaces and blank lines | collapse them |

> ⚠️ Notice what we do **not** touch: the long dash `–` in "0–1 years". It is a perfectly valid character. A rule like "delete all non-ASCII characters" would have damaged the text, and would destroy any Persian text completely.

In [3]:
import re
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("(cid:127)", "\n•")
    text = re.sub(r"Full[n■]time", "Full-time", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

before = parsed_documents[2].page_content
after = clean_text(before)

print("BEFORE:")
print(before[:250])
print()
print("AFTER:")
print(after[:250])

BEFORE:
Fullntime employees accrue PTO according to the following schedule: (cid:127) 0–1 years of service: 10 days per year (0.833 days per month) (cid:127) 1–3 years of service: 15 days per year (1.25 days per month) (cid:127) 3+ years of service: 20 days 

AFTER:
Full-time employees accrue PTO according to the following schedule:
• 0–1 years of service: 10 days per year (0.833 days per month)
• 1–3 years of service: 15 days per year (1.25 days per month)
• 3+ years of service: 20 days per year (1.67 days per 


In [4]:
cleaned_documents = [
    Document(page_content=clean_text(doc.page_content), metadata=doc.metadata)
    for doc in parsed_documents
]

for before_doc, after_doc in zip(parsed_documents, cleaned_documents):
    print(f'{after_doc.metadata["section"]}: {len(before_doc.page_content)} -> {len(after_doc.page_content)} characters')

Non-Disclosure Agreement (NDA) Policy: 427 -> 427 characters
Workplace Conduct Policy: 360 -> 360 characters
Paid Time Off (PTO) Policy: 582 -> 558 characters
Travel & Expense Policy: 462 -> 462 characters


## 🗑️ Step 3: Repeated headers and footers

Our PDF has one page, so it has no repeating headers. But real reports do: "ACME Corp - Confidential" at the top and "Page 7" at the bottom of **every page**. Those lines are noise: they would be searched and sent to the model again and again, with no useful meaning.

The idea: **a line that appears on most pages is probably a header or footer.**

Two details make this tricky:
- "Page 1", "Page 2", "Page 3" are all *different* strings. We replace digits with `#` before counting, so they match as `Page #`.
- With only 1 or 2 pages, "appears on most pages" means nothing (every line on a 1-page document appears on "100% of pages"). So we do nothing when there are too few pages.

Since our PDF can't show this, we test the function on a small made-up document.

In [5]:
from collections import Counter

def remove_repeated_lines(pages, min_repeat_ratio=0.6, min_pages=3):
    if len(pages) < min_pages:
        return pages, set()

    def key(line):
        return re.sub(r"\d+", "#", line.strip())

    counts = Counter()
    for page in pages:
        for k in {key(line) for line in page.split("\n") if line.strip()}:
            counts[k] += 1

    repeated = {k for k, n in counts.items() if n / len(pages) >= min_repeat_ratio}
    cleaned = [
        "\n".join(line for line in page.split("\n") if key(line) not in repeated)
        for page in pages
    ]
    return cleaned, repeated

fake_pages = [
    "ACME Corp - Confidential\nPTO gives 15 days.\nPage 1",
    "ACME Corp - Confidential\nRemote work is 3 days a week.\nPage 2",
    "ACME Corp - Confidential\nLaptops are provided by IT.\nPage 3",
    "ACME Corp - Confidential\nExpenses are paid in 30 days.\nPage 4",
]

fake_cleaned, repeated = remove_repeated_lines(fake_pages)
print("removed:", repeated)
print(fake_cleaned)

# a 1-page document must stay untouched
one_page, removed = remove_repeated_lines(["only page\nsecond line"])
print(one_page, removed)

removed: {'ACME Corp - Confidential', 'Page #'}
['PTO gives 15 days.', 'Remote work is 3 days a week.', 'Laptops are provided by IT.', 'Expenses are paid in 30 days.']
['only page\nsecond line'] set()


### 💡 Why this version works (and the old one didn't)

`min_repeat_ratio` is a **fraction between 0 and 1**: the share of pages a line must appear on. A line can appear on at most 100% of pages, so a ratio above 1 (like `1.5`) can *never* match, and nothing would ever be removed. Our `0.6` means "on at least 60% of pages".

⚠️ This rule is a **guess**, not a fact. A legitimate line that repeats on many pages (for example a table header) would also be removed. Always print what was removed (`repeated`) and check it.

## 📌 Summary

- **Cleaning** fixes defects in extracted text, so search and answers aren't built on broken text.
- Always **look first**: we searched for defects (`(cid:127)`, `Fullntime`) before writing any rule.
- One defect = one small, readable **rule**. We did *not* delete all non-ASCII characters, because the long dash `–` was valid.
- **Repeated headers/footers** can be found by counting lines that appear on most pages. Digits are replaced by `#` so "Page 1" and "Page 2" match.
- `min_repeat_ratio` is a fraction **between 0 and 1**. Values above 1 never match. Documents with too few pages must be left alone.
- Every rule is a guess. Print what you removed and check it.

## ⚠️ What cleaning can't do

- **It can't restore lost structure.** Look at the end of the PTO section: the sentence after the last bullet is still glued to it, because that line break was already lost in Lesson 03.
- **Our rules fit this one document.** `Full[n■]time` was written for what we saw. A different PDF will have different defects, so rules must come from *inspecting your own data*.
- **We only tested English.** Persian text has its own issues (for example different forms of the same letter, and invisible characters). We'll deal with those when we build the Persian version of the course.

## ✍️ Exercise (10 minutes)

Experiment with `min_repeat_ratio` on `fake_pages`:

1. Call `remove_repeated_lines(fake_pages, min_repeat_ratio=1.5)`. What is removed? Why?
2. Call it with `min_repeat_ratio=0.2`. What gets removed now, and why is that dangerous?

**Success:** you can explain in one sentence why the value must be chosen carefully.

💡 Solution: `solutions/04_cleaning.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Write a function `report_defects(documents)` that counts, across all documents:
- how many `(cid:N)` codes there are,
- which non-ASCII characters appear and how often.

Run it on `parsed_documents` and on `cleaned_documents`. A good cleaning step should make the first number go to zero. Why is such a *report* more useful than just cleaning silently?

## 📖 Next

Our text is clean and organized in sections. Next, we attach **metadata** to every piece, so later we can filter by it and cite where an answer came from.

**➡️ Next lesson:** [05 Metadata](05_metadata.ipynb)